
Capstone Project: Deep Learning for NLP

CONTRIBUTION - INDIVIDUAL

GITHUB -

In this capstone project, students will embark on an exciting journey to create an Industry-Specific Large Language Model (LLM) Bot using state-of-the-art pre-trained models from sources like Hugging Face. The primary objective is to build an intelligent bot that can effectively engage with users by answering questions and providing insights specific to a chosen industry. This project will not only enhance your technical skills but also provide a deep understanding of the chosen industry's nuances, challenges, and trends.

In this project, we developed an industry-specific chatbot for the retail banking sector using a fine-tuned large language model (LLM). The primary goal was to create an AI assistant capable of answering real customer queries about banking services such as account balance checks, card activation, ATM location, loan applications, and more.

We used the publicly available Bitext Retail Banking Chatbot Dataset, which includes thousands of real-world banking prompts and professional, human-like responses. The model was fine-tuned using Google’s FLAN-T5-small on cleaned Q&A pairs, and deployed using a Gradio web interface for user-friendly interaction.

This chatbot demonstrates how LLMs can be effectively adapted to specialized industries like banking, delivering fast, informative, and polite responses — similar to a live customer support agent.



In [1]:
pip install transformers datasets scikit-learn gradio


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("/content/drive/My Drive/AlmaBetter/Specialization_Track/Deep Learning for NLP/bitext-retail-banking-llm-chatbot-training-dataset.csv")

# Keep only needed columns and drop missing values
df_clean = df[['instruction', 'response']].dropna()
df_clean = df_clean[df_clean['instruction'].str.strip() != ""]
df_clean = df_clean[df_clean['response'].str.strip() != ""]
df_clean = df_clean.drop_duplicates(subset=['instruction', 'response'])

# Rename for model input/output
df_clean = df_clean.rename(columns={"instruction": "input", "response": "output"})

# Preview
df_clean.sample(3)


,input,output
7472,icant pay for my mortgage i need help cancelli...,I'm sorry to hear that you're facing difficult...
3007,need to apply for a mortgage how do i do it,Thank you for considering applying for a mortg...
15830,i dont use my ufcking account how to do it,I'm sincerely sorry for any inconvenience you ...


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_clean, test_size=0.1, random_state=42)


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
from datasets import Dataset

# Convert to Hugging Face format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function
def tokenize(example):
    input_enc = tokenizer(example["input"], padding="max_length", truncation=True, max_length=128)
    target_enc = tokenizer(example["output"], padding="max_length", truncation=True, max_length=128)
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

# Tokenize datasets
train_tokenized = train_dataset.map(tokenize)
test_tokenized = test_dataset.map(tokenize)


Map:   0%|          | 0/22990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2555 [00:00<?, ? examples/s]

In [7]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./banking_bot",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    num_train_epochs=3,
    logging_dir='./logs',
    eval_strategy="epoch"
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=collator
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,2.511721,2.075465
2,1.581708,1.386943
3,1.396175,1.261282


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17244, training_loss=2.5864894232487794, metrics={'train_runtime': 2732.7636, 'train_samples_per_second': 25.238, 'train_steps_per_second': 6.31, 'total_flos': 3205217027358720.0, 'train_loss': 2.5864894232487794, 'epoch': 3.0})

In [8]:
# prompt: give testing accuracy training accuracy and validation accuracy

# After trainer.train()
import json

# Function to extract metrics from logs
def get_metrics(log_history):
    train_loss = None
    eval_loss = None
    # Find the final training loss
    for log in log_history:
        if 'loss' in log:
            train_loss = log['loss']
        if 'eval_loss' in log:
            eval_loss = log['eval_loss']
    return train_loss, eval_loss

train_loss, eval_loss = get_metrics(trainer.state.log_history)

print(f"Training Loss (last logged): {train_loss}")
print(f"Validation Loss (last epoch): {eval_loss}")

# To get evaluation metrics on the test set after training
eval_results = trainer.evaluate(test_tokenized)
print(f"Test Evaluation Results: {eval_results}")

# Note: For accuracy in seq2seq models, it's usually not a single number
# like classification accuracy. Metrics like ROUGE, BLEU are common.
# If you need specific accuracy-like metrics, you would need to implement
# or use a library for sequence generation evaluation metrics and potentially
# override the compute_metrics function in the Trainer or perform manual evaluation.
# The default evaluation in Seq2SeqTrainer provides loss.



Training Loss (last logged): 1.3961746826171875
Validation Loss (last epoch): 1.2612816095352173


Test Evaluation Results: {'eval_loss': 1.2612816095352173, 'eval_runtime': 21.5558, 'eval_samples_per_second': 118.529, 'eval_steps_per_second': 29.644, 'epoch': 3.0}


In [9]:
model.save_pretrained("banking-chatbot")
tokenizer.save_pretrained("banking-chatbot")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('banking-chatbot/tokenizer_config.json', 'banking-chatbot/tokenizer.json')

In [17]:
import torch

def chat(prompt):
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Tokenize and move tensors to the same device as model
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate response
    with torch.no_grad():
        output = model.generate(**inputs, max_length=300)

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [18]:
chat("How can I activate my debit card?")


'I\'m here to assist you with activating your card. To activate your card, you can follow these steps: 1. Log in to your online banking account or mobile banking app. 2. Navigate to the "Account" or "Account" section. 3. Look for the option to activate your card. 4. Click on it to activate your card. 5. You should be able to view the activation details, such as your card number, account number, and any other necessary information. If you encounter any difficulties or have any further questions, please don\'t hesitate to reach out to our customer support team. They will be able to provide you with the necessary information, such as your card number,'

In [12]:
import gradio as gr

In [23]:
import gradio as gr

# Define custom CSS to explicitly set the minimum height for textareas
# Using !important to ensure it overrides any default Gradio styles
custom_css = """
.gr-box textarea {
  min-height: 200px !important; /* Adjust this value as needed */
}
"""

interface = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Your banking question", lines=5), # Keep lines=5 as a good default
    outputs=gr.Textbox(label="Chatbot response", lines=5),    # Keep lines=5 as a good default
    title="Retail Banking Chatbot",
    css=custom_css # Apply the custom CSS here
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2026e33338623636f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
